# Crop Disease Detection - Interactive UI
Upload images and get disease predictions

In [ ]:
import cv2
import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
from ipywidgets import widgets, interact
from IPython.display import display, clear_output

MODEL_PATH = Path(r'c:\Users\Administrator\Desktop\AIC\models\crop_disease_model.pkl')
LABEL_PATH = Path(r'c:\Users\Administrator\Desktop\AIC\models\label_encoder.pkl')

## Load Model

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)
with open(LABEL_PATH, 'rb') as f:
    labels = pickle.load(f)

print(f"Model loaded! Classes: {len(labels)}")

## Prediction Function

In [ ]:
def predict_disease(image_path):
    # Load image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Preprocess
    img_resized = cv2.resize(img_rgb, (64, 64))
    img_flat = img_resized.flatten() / 255.0
    img_input = img_flat.reshape(1, -1)
    
    # Predict
    pred_proba = model.predict_proba(img_input)[0]
    class_idx = np.argmax(pred_proba)
    confidence = pred_proba[class_idx]
    
    # Display results
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    # Show image
    ax[0].imshow(img_rgb)
    ax[0].set_title('Uploaded Image')
    ax[0].axis('off')
    
    # Show top 5 predictions
    top5_idx = np.argsort(pred_proba)[-5:][::-1]
    top5_labels = [labels[i] for i in top5_idx]
    top5_probs = [pred_proba[i] for i in top5_idx]
    
    ax[1].barh(top5_labels, top5_probs, color='steelblue')
    ax[1].set_xlabel('Confidence')
    ax[1].set_title('Top 5 Predictions')
    ax[1].set_xlim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n🎯 PREDICTION: {labels[class_idx]}")
    print(f"📊 CONFIDENCE: {confidence:.2%}")
    print(f"\nTop 3 Predictions:")
    for i in range(min(3, len(top5_idx))):
        print(f"  {i+1}. {top5_labels[i]}: {top5_probs[i]:.2%}")

## Test with Sample Images

Replace the path below with your test image path:

In [ ]:
# Example: Test with a sample image
test_image = r'c:\Users\Administrator\Desktop\AIC\plantvillage dataset\color\Apple___Apple_scab\00075aa8-d81a-4184-8541-b692b78d398a___FREC_Scab 3417.JPG'

predict_disease(test_image)

## Interactive File Selector

In [ ]:
# Create text input for image path
image_path_input = widgets.Text(
    value='',
    placeholder='Enter image path here',
    description='Image Path:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

predict_button = widgets.Button(
    description='Predict Disease',
    button_style='success',
    icon='check'
)

output = widgets.Output()

def on_predict_click(b):
    with output:
        clear_output()
        if image_path_input.value:
            try:
                predict_disease(image_path_input.value)
            except Exception as e:
                print(f"Error: {e}")
        else:
            print("Please enter an image path")

predict_button.on_click(on_predict_click)

display(image_path_input)
display(predict_button)
display(output)

## Batch Prediction

Test multiple images at once:

In [ ]:
import pandas as pd

# Load test images from CSV
df = pd.read_csv(r'c:\Users\Administrator\Desktop\AIC\data\processed\dataset_metadata.csv')
test_samples = df.sample(5, random_state=42)

print("Testing 5 random images:\n")
for idx, row in test_samples.iterrows():
    print(f"\n{'='*60}")
    print(f"Actual: {row['class_label']}")
    print(f"{'='*60}")
    predict_disease(row['image_path'])